# OOPSIE: Final 4-Condition Benchmark Evaluation (v1.2.0-frozen)

**Protocol**: `OOPSIE-EVAL-PROTOCOL-v1.2.0-frozen`  
**Target Corpus**: `evaluation/benchmark.jsonl` (N=40 records, SHA-256: `1676a9bcb9160bd142483cb868b4b652c21e93ef9f29e48f56786312c4a942dd`)  
**Base Model**: `google/gemma-4-12b-it` (4-bit NF4 precision, `torch.bfloat16`)  
**Adapter Checkpoint**: `oopsie-pilot-adapter` (Step 44, Epoch 2)  

### Experimental Conditions:
- **Primary Comparison**:
  - **Condition A (Base Zero-Shot + Schema)**: Upstream chat template + `system_prompt_v2_enum_guided` + complete JSON schema.
  - **Condition B (Adapter Zero-Shot + Schema)**: Training-aligned `<|turn>model\n` + identical `system_prompt_v2_enum_guided` + complete JSON schema.
- **Secondary Comparisons**:
  - **Condition C (Base 1-Shot + Schema)**: Upstream chat template + Condition A prompt + fixed training exemplar (`syn-dep-001`).
  - **Condition D (Adapter Current Service Prompt)**: Training-aligned `<|turn>model\n` + current production `system_prompt_v2_enum_guided` **WITHOUT** raw schema block.

> **RESUMABILITY & SAFETY CONSTRAINTS**:
> - **Resumable Checkpointing**: Results are incrementally saved to `checkpoint_results.jsonl`. If disconnected, rerun seamlessly skips completed cases.
> - **Manifest Integrity**: Pre-run hashes (benchmark dataset, schema, prompt v2, adapter) must match before resuming.
> - **Condition D Smoke Test**: Verifies Condition D on N=2 development cases before touching benchmark data.
> - **Blinded Review Isolation**: Human review sheet randomizes systems (Alpha, Beta, Gamma, Delta); unblinding key is written to a separate file.

## Step 1: Install Pinned Dependencies Matching Pilot Environment

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U trl>=0.12.0 peft>=0.14.0 bitsandbytes>=0.45.0 accelerate datasets
print("Dependencies installed successfully.")

## Step 2: Unpack & Verify Versioned Evaluation Bundle (v1.2.0-four-conditions)

In [ ]:
import os
import sys
import time
import json
import shutil
import hashlib
import zipfile
from pathlib import Path

bundle_dir = Path("/kaggle/working/eval_bundle")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

# Locate uploaded evaluation bundle zip or unpacked directory (prioritize v1.2.0 / four-conditions)
all_zips = list(Path("/kaggle/input").rglob("oopsie-eval-bundle*.zip")) + list(Path("/kaggle/working").rglob("oopsie-eval-bundle*.zip"))
v12_zips = [z for z in all_zips if "1.2" in z.name or "four" in z.name]
zip_candidates = v12_zips if v12_zips else all_zips

all_unpacked = [p.parent.parent for p in Path("/kaggle/input").rglob("evaluation/harness.py")]
v12_unpacked = [p for p in all_unpacked if "1.2" in str(p) or "four" in str(p)]
unpacked_candidates = v12_unpacked if v12_unpacked else all_unpacked

if zip_candidates:
    b_zip = zip_candidates[0]
    print(f"Extracting bundle from {b_zip}...")
    with zipfile.ZipFile(b_zip, "r") as zf:
        zf.extractall(bundle_dir)
elif unpacked_candidates:
    b_dir = unpacked_candidates[0]
    print(f"Copying unpacked bundle from {b_dir}...")
    shutil.copytree(b_dir, bundle_dir, dirs_exist_ok=True)
else:
    raise FileNotFoundError("No oopsie-eval-bundle found in /kaggle/input or /kaggle/working!")

sys.path.insert(0, str(bundle_dir))
print("Bundle loaded on sys.path. Verifying imports...")

import evaluation.harness as harness
if not hasattr(harness, 'CONDITIONS'):
    raise AttributeError(
        "ERROR: Kaggle loaded an older v1.1.0 bundle which lacks the 4-condition specification and benchmark dataset!\n"
        "Please upload 'oopsie-eval-bundle-v1.2.0-four-conditions.zip' to Kaggle and remove the older v1.1.0 dataset."
    )

hashes = harness.verify_integrity_hashes()
print("Verified integrity hashes:")
for k, v in hashes.items():
    print(f"  {k:22}: {v}")

# Verify the 4 conditions exist
print(f"\nRegistered evaluation conditions: {list(harness.CONDITIONS.keys())}")
for cid, cond in harness.CONDITIONS.items():
    print(f"  [{cid}] {cond.name} (use_adapter={cond.use_adapter}, schema={cond.include_schema_in_prompt})")

## Step 3: Load Gemma 4 12B Base Model in 4-bit NF4 Quantization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL_ID = "google/gemma-4-12b-it"
print(f"Loading base model: {BASE_MODEL_ID} in 4-bit NF4 (compute_dtype=torch.bfloat16)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
load_time = time.perf_counter() - t0

vram_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
vram_res = torch.cuda.memory_reserved() / (1024 ** 3)
print(f"Base model loaded in {load_time:.2f}s.")
print(f"VRAM allocated: {vram_alloc:.2f} GB, reserved: {vram_res:.2f} GB")

## Step 4: Attach Fine-Tuned Pilot Adapter Checkpoint-44

In [ ]:
from peft import PeftModel

# Locate adapter directory
adapter_candidates = [
    p for p in Path("/kaggle/input").rglob("adapter_config.json")
    if "checkpoint" in str(p).lower() or "pilot" in str(p).lower() or "adapter" in str(p).lower()
]

if adapter_candidates:
    adapter_path = adapter_candidates[0].parent
else:
    # Fallback to local working directory if uncompressed
    adapter_path = Path("/kaggle/working/oopsie-pilot-adapter")

print(f"Attaching PEFT adapter from: {adapter_path}...")
model = PeftModel.from_pretrained(base_model, str(adapter_path), adapter_name="oopsie_pilot")
model.eval()
print("PEFT adapter attached successfully.")

## Step 5: Verify PEFT Runtime Adapter State Management

In [ ]:
# 1. Initial adapted state
st_init = harness.inspect_peft_layer_status(model, expected_adapter="oopsie_pilot")
print("Initial Adapted State:")
for k, v in st_init.items():
    print(f"  {k:20}: {v}")
assert st_init["enabled"] is True, "Adapters must be initially enabled!"
assert st_init["any_merged"] is False, "LoRA weights must not be merged!"

# 2. Disabled state inside context manager
with model.disable_adapter():
    st_disabled = harness.inspect_peft_layer_status(model, expected_adapter="oopsie_pilot")
    print("\nInside disable_adapter() context:")
    for k, v in st_disabled.items():
        print(f"  {k:20}: {v}")
    assert st_disabled["none_enabled"] is True, "All layers must be disabled inside context!"
    assert st_disabled["any_merged"] is False, "LoRA weights must not be merged!"

# 3. Restoration after context exit
st_restored = harness.inspect_peft_layer_status(model, expected_adapter="oopsie_pilot")
print("\nAfter context exit:")
for k, v in st_restored.items():
    print(f"  {k:20}: {v}")
assert st_restored["enabled"] is True, "Adapters must be restored after context exit!"
print("\nAll PEFT adapter layer status assertions PASSED.")

## Step 6: Development Smoke Test — Condition D Verification (N=2 Cases from Validation Split)

Runs a small fixed smoke test on 2 development cases (`syn-amb-020` and `syn-ath-055`) under Condition D before touching benchmark data to verify prompt identity, adapter state, formatter, output capture, and scoring completeness.

In [ ]:
val_path = bundle_dir / "data" / "validation.jsonl"
val_records = [json.loads(l) for l in val_path.read_text(encoding="utf-8").splitlines() if l.strip()]
smoke_cases = [r for r in val_records if r["id"] in ("syn-amb-020", "syn-ath-055")]

smoke_out_dir = Path("/kaggle/working/oopsie-smoke-test-d")
smoke_out_dir.mkdir(parents=True, exist_ok=True)

print(f"Running Condition D development smoke test on {len(smoke_cases)} cases...")
smoke_results = {}
for sc in smoke_cases:
    cid = sc["id"]
    res_d = harness.evaluate_single_case(sc, harness.CONDITION_D, model, tokenizer)
    smoke_results[cid] = res_d
    print(f"  [{cid}] Cond D -> JSON valid: {res_d.json_valid}, Schema valid: {res_d.schema_valid}, Budget hit: {res_d.hit_budget}, Tokens: {res_d.gen_token_count}")

with open(smoke_out_dir / "smoke_d_results.json", "w", encoding="utf-8") as f:
    json.dump({k: harness.asdict(v) for k, v in smoke_results.items()}, f, indent=2)

print("Condition D smoke test complete. Infrastructure and scoring verified.")

## Step 7: Load 40 Reserved Benchmark Records & Verify Input Contract

In [ ]:
bench_path = bundle_dir / "evaluation" / "benchmark.jsonl"
bench_bytes = bench_path.read_bytes()
actual_bench_hash = hashlib.sha256(bench_bytes).hexdigest()
EXPECTED_BENCH_HASH = "1676a9bcb9160bd142483cb868b4b652c21e93ef9f29e48f56786312c4a942dd"

print(f"Benchmark path: {bench_path}")
print(f"Benchmark SHA-256: {actual_bench_hash}")
assert actual_bench_hash == EXPECTED_BENCH_HASH, f"Benchmark hash mismatch! Expected {EXPECTED_BENCH_HASH}, got {actual_bench_hash}"

bench_records = [json.loads(l) for l in bench_path.read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"Loaded {len(bench_records)} reserved benchmark records.")
assert len(bench_records) == 40, f"Expected 40 records, got {len(bench_records)}"

# Verify prompt rendering for sample record across all 4 conditions
sample_rec = bench_records[0]
print(f"\nSample Record ID: {sample_rec['id']}")
for ckey in ["cond_a", "cond_b", "cond_c", "cond_d"]:
    cond = harness.CONDITIONS[ckey]
    p_str, msgs = harness.render_prompt_for_condition(sample_rec, cond, tokenizer)
    p_toks = len(tokenizer(p_str)["input_ids"])
    print(f"  [{ckey}] {cond.name[:45]:45}: {p_toks:4d} tokens (Hash: {hashlib.sha256(p_str.encode('utf-8')).hexdigest()[:12]}...)")

## Step 8: Pre-Run Checkpoint Manifest & Resumption Setup

Enables incremental, crash-resilient evaluation across all 40 cases × 4 conditions (160 unique evaluations).

In [ ]:
out_dir = Path("/kaggle/working/oopsie-benchmark-results-v1-2-0")
out_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = out_dir / "checkpoint_results.jsonl"
manifest_path = out_dir / "run_manifest.json"

# Write or verify pre-run manifest
manifest_data = {
    "protocol": "OOPSIE-EVAL-PROTOCOL-v1.2.0-frozen",
    "benchmark_records": 40,
    "benchmark_sha256": EXPECTED_BENCH_HASH,
    "canonical_schema_sha256": harness.EXPECTED_SCHEMA_HASH,
    "system_prompt_v2_sha256": harness.EXPECTED_PROMPT_V2_HASH,
    "base_model_id": BASE_MODEL_ID,
    "adapter_checkpoint": "oopsie-pilot-adapter (Step 44)",
    "precision": "4-bit NF4, torch.bfloat16",
    "decoding": "greedy (temp=0.0, do_sample=False, max_new_tokens=1024)",
    "conditions": {
        ck: {
            "name": c.name,
            "use_adapter": c.use_adapter,
            "formatter_id": c.formatter_id,
            "include_schema": c.include_schema_in_prompt,
            "demo": c.include_demonstration,
            "prompt_ver": c.system_prompt_version,
        }
        for ck, c in harness.CONDITIONS.items()
    },
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}

if not manifest_path.exists():
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)
    print("Pre-run manifest written.")
else:
    # Validate existing manifest matches current execution
    existing_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert existing_manifest["benchmark_sha256"] == EXPECTED_BENCH_HASH, "Cannot resume: Benchmark hash mismatch!"
    assert existing_manifest["canonical_schema_sha256"] == harness.EXPECTED_SCHEMA_HASH, "Cannot resume: Schema hash mismatch!"
    assert existing_manifest["system_prompt_v2_sha256"] == harness.EXPECTED_PROMPT_V2_HASH, "Cannot resume: System prompt hash mismatch!"
    print("Pre-run manifest validated. Resuming under matching configuration.")

# Load existing checkpoints using hardened recovery logic
completed_passes = set()
bench_results_by_case = {r["id"]: {} for r in bench_records}

completed_dict, incomplete_list, audit_warnings = harness.load_checkpoint_records(checkpoint_path)
for (cid, ckey), res_obj in completed_dict.items():
    if cid in bench_results_by_case:
        bench_results_by_case[cid][ckey] = res_obj
        completed_passes.add((cid, ckey))

if incomplete_list:
    print(f"\n[RESUMPTION NOTICE] Isolated {len(incomplete_list)} incomplete attempt(s) from earlier interrupted writes:")
    for inc in incomplete_list:
        print(f"  Line {inc.get('line_num')}: {inc.get('error', 'unknown error')}")

if audit_warnings:
    print(f"\nCheckpoint audit warnings logged ({len(audit_warnings)}):")
    for w in audit_warnings[:5]:
        print(f"  - {w}")

print(f"Checkpoint status: {len(completed_passes)} / 160 case-condition evaluations verified and ready.")

## Step 9: Resumable 4-Condition Benchmark Evaluation Loop (160 Unique Passes)

Executes all 40 cases across Conditions A, B, C, and D with immediate per-case checkpointing.

In [ ]:
fixed_demo = harness.load_fixed_demonstration()
total_target = 40 * len(harness.CONDITIONS)

print(f"Beginning evaluation loop across {len(bench_records)} cases x 4 conditions ({total_target} total passes)...")
print("Estimated runtime based on dev dry-run latency: ~4.5 - 5.5 hours.\n")

ckpt_file = open(checkpoint_path, "a", encoding="utf-8")

try:
    for i, rec in enumerate(bench_records, 1):
        cid = rec["id"]
        print(f"\n[{i:02d}/40] Case: {cid} (Category: {rec.get('expected_category')})")
        
        for ckey in ["cond_a", "cond_b", "cond_c", "cond_d"]:
            if (cid, ckey) in completed_passes:
                prior_res = bench_results_by_case[cid][ckey]
                print(f"  [{ckey}] (RESUMED) schema_valid: {prior_res.schema_valid}, budget_hit: {prior_res.hit_budget}, tok: {prior_res.gen_token_count}")
                continue
            
            cond = harness.CONDITIONS[ckey]
            res = harness.evaluate_single_case(rec, cond, model, tokenizer, fixed_demo=fixed_demo)
            bench_results_by_case[cid][ckey] = res
            completed_passes.add((cid, ckey))
            
            # Incremental checkpoint write with flush
            ckpt_record = {
                "case_id": cid,
                "condition_key": ckey,
                "condition_name": cond.name,
                "result": harness.asdict(res)
            }
            ckpt_file.write(json.dumps(ckpt_record) + "\n")
            ckpt_file.flush()
            
            print(f"  [{ckey}] schema_valid: {res.schema_valid}, json_valid: {res.json_valid}, budget_hit: {res.hit_budget}, dur: {res.duration_seconds:.1f}s, tok: {res.gen_token_count}")

finally:
    ckpt_file.close()

print(f"\nEvaluation loop finished. Completed passes: {len(completed_passes)} / {total_target}.")

## Step 10: Comparative Metrics Summary, 2x2 Paired Transitions & Confidence Intervals

In [ ]:
import numpy as np

def wilson_score_interval(successes, total, z=1.96):
    if total == 0:
        return 0.0, 0.0
    p = successes / total
    denom = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denom
    spread = z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denom
    return max(0.0, center - spread), min(1.0, center + spread)

def summarize_condition(cond_key, name):
    cases = list(bench_results_by_case.keys())
    tot = len(cases)
    res_list = [bench_results_by_case[c].get(cond_key) for c in cases if cond_key in bench_results_by_case[c]]
    n_done = len(res_list)
    
    jv = sum(1 for r in res_list if r.json_valid)
    sv = sum(1 for r in res_list if r.schema_valid)
    hb = sum(1 for r in res_list if r.hit_budget)
    gf = sum(1 for r in res_list if r.generation_failed)
    cm_all = sum(1 for r in res_list if r.category_match is True)
    cm_parse = sum(1 for r in res_list if r.schema_valid and r.category_match is True)
    sp = sum(1 for r in res_list if r.safety_audit.get("safety_pass") is True)
    avg_dur = np.mean([r.duration_seconds for r in res_list]) if n_done else 0.0
    
    ci_low, ci_high = wilson_score_interval(sv, n_done)
    
    return {
        "condition": name,
        "completed": f"{n_done}/{tot}",
        "strict_schema_valid": f"{sv}/{n_done} ({sv/n_done*100:.1f}%)" if n_done else "N/A",
        "schema_valid_95ci": f"[{ci_low*100:.1f}%, {ci_high*100:.1f}%]" if n_done else "N/A",
        "raw_json_valid": f"{jv}/{n_done} ({jv/n_done*100:.1f}%)" if n_done else "N/A",
        "budget_terminations": f"{hb}/{n_done} ({hb/n_done*100:.1f}%)" if n_done else "N/A",
        "category_agreement_all": f"{cm_all}/{n_done} ({cm_all/n_done*100:.1f}%)" if n_done else "N/A",
        "category_agreement_parseable": f"{cm_parse}/{sv} ({cm_parse/sv*100:.1f}%)" if sv else "N/A",
        "heuristic_safety_pass": f"{sp}/{n_done} ({sp/n_done*100:.1f}%)" if n_done else "N/A",
        "generation_failures": f"{gf}/{n_done}",
        "avg_duration_seconds": round(float(avg_dur), 2)
    }

summary = [
    summarize_condition("cond_a", "Condition A (Base Zero-Shot + Schema)"),
    summarize_condition("cond_b", "Condition B (Adapter Zero-Shot + Schema)"),
    summarize_condition("cond_c", "Condition C (Base 1-Shot + Schema)"),
    summarize_condition("cond_d", "Condition D (Adapter Current Service Prompt)")
]

print("=== BENCHMARK EVALUATION SUMMARY METRICS ===")
for s in summary:
    print(f"\n--- {s['condition']} ---")
    for k, v in s.items():
        if k != 'condition':
            print(f"  {k:30}: {v}")

# Paired 2x2 contingency table helper
def print_paired_2x2(k1, k2, n1, n2):
    cases = [c for c in bench_results_by_case if k1 in bench_results_by_case[c] and k2 in bench_results_by_case[c]]
    tot = len(cases)
    both_p = sum(1 for c in cases if bench_results_by_case[c][k1].schema_valid and bench_results_by_case[c][k2].schema_valid)
    k2_only = sum(1 for c in cases if not bench_results_by_case[c][k1].schema_valid and bench_results_by_case[c][k2].schema_valid)
    k1_only = sum(1 for c in cases if bench_results_by_case[c][k1].schema_valid and not bench_results_by_case[c][k2].schema_valid)
    both_f = sum(1 for c in cases if not bench_results_by_case[c][k1].schema_valid and not bench_results_by_case[c][k2].schema_valid)
    print(f"\n=== PAIRED 2x2 TRANSITIONS: {n1} vs {n2} (N={tot}) ===")
    print(f"  Both Pass (+ +)             : {both_p} / {tot}")
    print(f"  {n2}-Only Pass (- +)         : {k2_only} / {tot}")
    print(f"  {n1}-Only Pass (+ -)         : {k1_only} / {tot}")
    print(f"  Both Fail (- -)             : {both_f} / {tot}")

print_paired_2x2("cond_a", "cond_b", "Base Zero-Shot (A)", "Adapter Zero-Shot (B)")
print_paired_2x2("cond_b", "cond_d", "Adapter Matched (B)", "Adapter Service (D)")
print_paired_2x2("cond_a", "cond_c", "Base Zero-Shot (A)", "Base 1-Shot (C)")

## Step 11: Export Full Artifacts, Blinded Review Sheet & Downloadable Tarball

In [ ]:
import tarfile
from IPython.display import FileLink

# 1. Save full records JSON
full_json_path = out_dir / "benchmark_results_full.json"
with open(full_json_path, "w", encoding="utf-8") as f:
    json.dump({
        cid: {k: harness.asdict(v) for k, v in cmap.items()}
        for cid, cmap in bench_results_by_case.items()
    }, f, indent=2)

# 2. Save summary metrics
summary_path = out_dir / "summary_metrics.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

# 3. Generate Blinded Review Sheet and Separate Unblinding Key
review_sheet_path = out_dir / "blinded_human_review_sheet.md"
key_path = out_dir / "unblinded_review_key.json"
harness.generate_blinded_review_sheet(
    bench_results_by_case,
    {r["id"]: r for r in bench_records},
    review_sheet_path,
    key_output_path=key_path,
    seed=42,
)

# 4. Write execution manifest with checksums
exec_manifest = {
    "run_id": "oopsie-benchmark-final-v1-2-0",
    "protocol": "OOPSIE-EVAL-PROTOCOL-v1.2.0-frozen",
    "benchmark_records": len(bench_records),
    "files": {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest()
        for p in [full_json_path, summary_path, review_sheet_path, key_path, checkpoint_path]
        if p.exists()
    }
}
with open(out_dir / "execution_manifest.json", "w", encoding="utf-8") as f:
    json.dump(exec_manifest, f, indent=2)

# 5. Create downloadable tarball in /kaggle/working
tar_path = Path("/kaggle/working/oopsie_benchmark_results_v1_2_0.tar.gz")
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(out_dir, arcname="benchmark_results")

tar_hash = hashlib.sha256(tar_path.read_bytes()).hexdigest()
print(f"\nAll benchmark artifacts successfully exported to {out_dir}")
print(f"Downloadable tarball created at: {tar_path} (SHA-256: {tar_hash})")
FileLink(str(tar_path))